El script debe dar tratamiento conjuntos de datos grandes

  1.​ Tomar muestras de 5.000 y 10.000 registros.

  2.​ Calcular la matriz de distancias Gower para cada muestra.

  3.​ Medir tiempo de ejecución y uso de memoria.
  
  4.​ Comparar estabilidad de clusters en muestras de distinto tamaño.​

Muestra de 5000 registros:

In [1]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
import time # Para medir el tiempo de ejecución
import psutil # Para medir el uso de memoria
import os # Para obtener el ID del proceso actual

# Configuración para mejor visualización en Jupyter
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

# =================================================================
# Carga del DataFrame Original (con valores faltantes)
# =================================================================
# Obtener el dataset "Adult" (id=2)
adult = fetch_ucirepo(id=2)

# Data como DataFrames de pandas
X = adult.data.features
y = adult.data.targets
df = pd.concat([X, y], axis=1)

# Reemplazar el caracter '?' por NaN de NumPy para un manejo estándar
df.replace('?', np.nan, inplace=True)

# =================================================================
# Implementación Vectorizada de Gower para una MUESTRA
# =================================================================

# 1. Preparación de los datos para Gower
# -----------------------------------------------------------------
# Define el tamaño de la muestra
sample_size = 5000

# Tomamos una muestra aleatoria del DataFrame
df_sample = df.sample(n=sample_size, random_state=42).copy()

print(f"--- Se ha tomado una muestra aleatoria de {sample_size} filas. ---")

# Identificar tipos de variables de forma definitiva
numeric_cols = ['age', 'fnlwgt', 'capital-gain', 'capital-loss', 'hours-per-week', 'education-num']
ordinal_cols = ['education']
categorical_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']

# --- Manejo de la Variable Ordinal 'education' ---
education_map = {
    'Preschool': 1, '1st-4th': 2, '5th-6th': 3, '7th-8th': 4, '9th': 5,
    '10th': 6, '11th': 7, '12th': 8, 'HS-grad': 9, 'Some-college': 10,
    'Assoc-voc': 11, 'Assoc-acdm': 12, 'Bachelors': 13, 'Masters': 14,
    'Prof-school': 15, 'Doctorate': 16
}
df['education_rank'] = df['education'].map(education_map)
min_rank = df['education_rank'].min()
max_rank = df['education_rank'].max()
ordinal_rank_range = max_rank - min_rank
df_sample['education'] = (df_sample['education'].map(education_map) - min_rank) / ordinal_rank_range

# Pre-calcular los rangos para las variables numéricas
ranges = {col: df[col].max() - df[col].min() for col in numeric_cols}


# 2. Cálculo Vectorizado de la Matriz de Distancias
# -----------------------------------------------------------------
print("\nCalculando la matriz de distancias (VERSIÓN VECTORIZADA)...")

# --- INICIO DE MEDICIÓN DE RECURSOS ---
process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 * 1024) # Memoria en MB
start_time = time.time()
print(f"Uso de memoria antes del cálculo: {mem_before:.2f} MB")
# ------------------------------------


# Paso A: Separar los datos en arrays de NumPy por tipo
df_num = df_sample[numeric_cols].to_numpy(dtype=np.float64)
df_cat = df_sample[categorical_cols].to_numpy(dtype=object)
df_ord = df_sample[ordinal_cols].to_numpy(dtype=np.float64)

# Paso B: Calcular la disimilitud para cada tipo de variable
numeric_dissim = np.sum([
    np.abs(np.subtract.outer(df_num[:, i], df_num[:, i])) / ranges[col]
    for i, col in enumerate(numeric_cols) if ranges[col] > 0
], axis=0)

categorical_dissim = np.sum([
    np.not_equal.outer(df_cat[:, i], df_cat[:, i]).astype(np.float64)
    for i in range(df_cat.shape[1])
], axis=0)

ordinal_dissim = np.sum([
    np.abs(np.subtract.outer(df_ord[:, i], df_ord[:, i]))
    for i in range(df_ord.shape[1])
], axis=0)

# Paso C: Sumar todas las disimilitudes (numerador de Gower)
total_dissimilarity = numeric_dissim + categorical_dissim + ordinal_dissim

# Paso D: Calcular los pesos (denominador de Gower) de forma vectorizada
is_not_nan = ~df_sample.drop(columns=['education_rank'], errors='ignore').isna()
weight_sum = np.dot(is_not_nan.astype(int), is_not_nan.astype(int).T)

# Paso E: Calcular la matriz de Gower final, evitando la división por cero
with np.errstate(divide='ignore', invalid='ignore'):
    gower_matrix_vectorized = np.where(weight_sum > 0, total_dissimilarity / weight_sum, 0)


# --- FIN DE MEDICIÓN DE RECURSOS ---
end_time = time.time()
mem_after = process.memory_info().rss / (1024 * 1024) # Memoria en MB

print("\n--- Resumen de Ejecución (Vectorizado) ---")
print(f"¡Cálculo completado en {end_time - start_time:.4f} segundos!")
print(f"Uso de memoria final: {mem_after:.2f} MB")
print(f"Memoria adicional utilizada por el cálculo: {mem_after - mem_before:.2f} MB")
# ---------------------------------


# 3. Visualización de Resultados
# -----------------------------------------------------------------
gower_matrix_df = pd.DataFrame(gower_matrix_vectorized, index=df_sample.index, columns=df_sample.index)
print("\n--- Matriz de Distancias de Gower resultante (versión vectorizada) ---")
print("(Mostrando las primeras 10x10 para visualización)")

# Retornar el DataFrame lo mostrará en formato tabla
gower_matrix_df.iloc[:10, :10]

--- Se ha tomado una muestra aleatoria de 5000 filas. ---

Calculando la matriz de distancias (VERSIÓN VECTORIZADA)...
Uso de memoria antes del cálculo: 137.74 MB

--- Resumen de Ejecución (Vectorizado) ---
¡Cálculo completado en 4.4071 segundos!
Uso de memoria final: 1304.82 MB
Memoria adicional utilizada por el cálculo: 1167.08 MB

--- Matriz de Distancias de Gower resultante (versión vectorizada) ---
(Mostrando las primeras 10x10 para visualización)


,7762,23881,30507,28911,19484,43031,28188,12761,40834,27875
7762,0.0000,0.2208,0.2900,0.2268,0.3257,0.3973,0.2419,0.3050,0.3392,0.3222
23881,0.2208,0.0000,0.3708,0.0291,0.3970,0.4826,0.3093,0.3663,0.3473,0.3231
30507,0.2900,0.3708,0.0000,0.3595,0.3690,0.3785,0.3519,0.4223,0.5159,0.3724
28911,0.2268,0.0291,0.3595,0.0000,0.3857,0.4535,0.2996,0.3727,0.3182,0.3132
19484,0.3257,0.3970,0.3690,0.3857,0.0000,0.4662,0.3544,0.3896,0.5127,0.4251
43031,0.3973,0.4826,0.3785,0.4535,0.4662,0.0000,0.1824,0.2674,0.3597,0.3362
28188,0.2419,0.3093,0.3519,0.2996,0.3544,0.1824,0.0000,0.1058,0.3714,0.2873
12761,0.3050,0.3663,0.4223,0.3727,0.3896,0.2674,0.1058,0.0000,0.4698,0.3929
40834,0.3392,0.3473,0.5159,0.3182,0.5127,0.3597,0.3714,0.4698,0.0000,0.3795
27875,0.3222,0.3231,0.3724,0.3132,0.4251,0.3362,0.2873,0.3929,0.3795,0.0000


Muestra de 10.000 registros:

In [2]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
import time # Para medir el tiempo de ejecución
import psutil # Para medir el uso de memoria
import os # Para obtener el ID del proceso actual

# Configuración para mejor visualización en Jupyter
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

# =================================================================
# Carga del DataFrame Original (con valores faltantes)
# =================================================================
# Obtener el dataset "Adult" (id=2)
adult = fetch_ucirepo(id=2)

# Data como DataFrames de pandas
X = adult.data.features
y = adult.data.targets
df = pd.concat([X, y], axis=1)

# Reemplazar el caracter '?' por NaN de NumPy para un manejo estándar
df.replace('?', np.nan, inplace=True)

# =================================================================
# Implementación Vectorizada de Gower para una MUESTRA
# =================================================================

# 1. Preparación de los datos para Gower
# -----------------------------------------------------------------
# Define el tamaño de la muestra
sample_size = 10000

# Tomamos una muestra aleatoria del DataFrame
df_sample = df.sample(n=sample_size, random_state=42).copy()

print(f"--- Se ha tomado una muestra aleatoria de {sample_size} filas. ---")

# Identificar tipos de variables de forma definitiva
numeric_cols = ['age', 'fnlwgt', 'capital-gain', 'capital-loss', 'hours-per-week', 'education-num']
ordinal_cols = ['education']
categorical_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']

# --- Manejo de la Variable Ordinal 'education' ---
education_map = {
    'Preschool': 1, '1st-4th': 2, '5th-6th': 3, '7th-8th': 4, '9th': 5,
    '10th': 6, '11th': 7, '12th': 8, 'HS-grad': 9, 'Some-college': 10,
    'Assoc-voc': 11, 'Assoc-acdm': 12, 'Bachelors': 13, 'Masters': 14,
    'Prof-school': 15, 'Doctorate': 16
}
df['education_rank'] = df['education'].map(education_map)
min_rank = df['education_rank'].min()
max_rank = df['education_rank'].max()
ordinal_rank_range = max_rank - min_rank
df_sample['education'] = (df_sample['education'].map(education_map) - min_rank) / ordinal_rank_range

# Pre-calcular los rangos para las variables numéricas
ranges = {col: df[col].max() - df[col].min() for col in numeric_cols}


# 2. Cálculo Vectorizado de la Matriz de Distancias
# -----------------------------------------------------------------
print("\nCalculando la matriz de distancias (VERSIÓN VECTORIZADA)...")

# --- INICIO DE MEDICIÓN DE RECURSOS ---
process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 * 1024) # Memoria en MB
start_time = time.time()
print(f"Uso de memoria antes del cálculo: {mem_before:.2f} MB")
# ------------------------------------


# Paso A: Separar los datos en arrays de NumPy por tipo
df_num = df_sample[numeric_cols].to_numpy(dtype=np.float64)
df_cat = df_sample[categorical_cols].to_numpy(dtype=object)
df_ord = df_sample[ordinal_cols].to_numpy(dtype=np.float64)

# Paso B: Calcular la disimilitud para cada tipo de variable
numeric_dissim = np.sum([
    np.abs(np.subtract.outer(df_num[:, i], df_num[:, i])) / ranges[col]
    for i, col in enumerate(numeric_cols) if ranges[col] > 0
], axis=0)

categorical_dissim = np.sum([
    np.not_equal.outer(df_cat[:, i], df_cat[:, i]).astype(np.float64)
    for i in range(df_cat.shape[1])
], axis=0)

ordinal_dissim = np.sum([
    np.abs(np.subtract.outer(df_ord[:, i], df_ord[:, i]))
    for i in range(df_ord.shape[1])
], axis=0)

# Paso C: Sumar todas las disimilitudes (numerador de Gower)
total_dissimilarity = numeric_dissim + categorical_dissim + ordinal_dissim

# Paso D: Calcular los pesos (denominador de Gower) de forma vectorizada
is_not_nan = ~df_sample.drop(columns=['education_rank'], errors='ignore').isna()
weight_sum = np.dot(is_not_nan.astype(int), is_not_nan.astype(int).T)

# Paso E: Calcular la matriz de Gower final, evitando la división por cero
with np.errstate(divide='ignore', invalid='ignore'):
    gower_matrix_vectorized = np.where(weight_sum > 0, total_dissimilarity / weight_sum, 0)


# --- FIN DE MEDICIÓN DE RECURSOS ---
end_time = time.time()
mem_after = process.memory_info().rss / (1024 * 1024) # Memoria en MB

print("\n--- Resumen de Ejecución (Vectorizado) ---")
print(f"¡Cálculo completado en {end_time - start_time:.4f} segundos!")
print(f"Uso de memoria final: {mem_after:.2f} MB")
print(f"Memoria adicional utilizada por el cálculo: {mem_after - mem_before:.2f} MB")
# ---------------------------------


# 3. Visualización de Resultados
# -----------------------------------------------------------------
gower_matrix_df = pd.DataFrame(gower_matrix_vectorized, index=df_sample.index, columns=df_sample.index)
print("\n--- Matriz de Distancias de Gower resultante (versión vectorizada) ---")
print("(Mostrando las primeras 10x10 para visualización)")

# Retornar el DataFrame lo mostrará en formato tabla
gower_matrix_df.iloc[:10, :10]

--- Se ha tomado una muestra aleatoria de 10000 filas. ---

Calculando la matriz de distancias (VERSIÓN VECTORIZADA)...
Uso de memoria antes del cálculo: 1325.68 MB

--- Resumen de Ejecución (Vectorizado) ---
¡Cálculo completado en 17.4099 segundos!
Uso de memoria final: 4949.80 MB
Memoria adicional utilizada por el cálculo: 3624.12 MB

--- Matriz de Distancias de Gower resultante (versión vectorizada) ---
(Mostrando las primeras 10x10 para visualización)


,7762,23881,30507,28911,19484,43031,28188,12761,40834,27875
7762,0.0000,0.2208,0.2900,0.2268,0.3257,0.3973,0.2419,0.3050,0.3392,0.3222
23881,0.2208,0.0000,0.3708,0.0291,0.3970,0.4826,0.3093,0.3663,0.3473,0.3231
30507,0.2900,0.3708,0.0000,0.3595,0.3690,0.3785,0.3519,0.4223,0.5159,0.3724
28911,0.2268,0.0291,0.3595,0.0000,0.3857,0.4535,0.2996,0.3727,0.3182,0.3132
19484,0.3257,0.3970,0.3690,0.3857,0.0000,0.4662,0.3544,0.3896,0.5127,0.4251
43031,0.3973,0.4826,0.3785,0.4535,0.4662,0.0000,0.1824,0.2674,0.3597,0.3362
28188,0.2419,0.3093,0.3519,0.2996,0.3544,0.1824,0.0000,0.1058,0.3714,0.2873
12761,0.3050,0.3663,0.4223,0.3727,0.3896,0.2674,0.1058,0.0000,0.4698,0.3929
40834,0.3392,0.3473,0.5159,0.3182,0.5127,0.3597,0.3714,0.4698,0.0000,0.3795
27875,0.3222,0.3231,0.3724,0.3132,0.4251,0.3362,0.2873,0.3929,0.3795,0.0000
